<a href="https://colab.research.google.com/github/buntyedutainer/student-performance-ai/blob/main/Notebook5_Split_Scaling_Leakage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Upload the file into Colab
from google.colab import files
files.upload()   # pick student-clean.csv

Saving student_clean.csv to student_clean.csv


{'student_clean.csv': b'school,sex,age,address,famsize,Pstatus,Medu,Fedu,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other\n0,0,18,1,1,0,4,4,2,2,0,1,0,0,0,1,1,0,0,4,3,4,1,1,3,6,5,6,6,0,0,0,0,0,0,0,1,0,0,0,1,0\n0,0,17,1,1,1,1,1,1,2,0,0,1,0,0,0,1,1,0,5,3,3,1,1,3,4,5,5,6,0,0,0,0,0,1,0,0,0,0,0,0,0\n0,0,15,1,0,1,1,1,1,2,3,1,0,1,0,1,1,1,0,4,3,2,2,3,3,10,7,8,10,0,0,0,0,0,1,0,0,0,1,0,1,0\n0,0,15,1,1,1,4,2,1,3,0,0,1,1,1,1,1,1,1,3,2,2,1,1,5,2,15,14,15,1,0,0,0,0,0,1,0,1,0,0,1,0\n0,0,16,1,1,1,3,3,1,2,0,0,1,1,0,1,1,0,0,4,3,2,1,2,5,4,6,10,10,0,1,0,0,0,1,0,0,1,0,0,0,0\n0,1,16,1,0,1,4,3,1,2,0,0,1,1,1,1,1,1,0,5,4,2,1,2,5,10,15,15,15,0,0,1,0,0,1,0,0,0,0,1,1,0\n0,1,16,1,0,1,2,2,1,2,0,0,0,0,0,1,1,1,0,4,4,4,1,1,3,0,12,12,11,0,1,0,0,0,

In [4]:
# Cell 1: setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("student_clean.csv")
print("Shape:", df.shape)       # about 357 rows: 395 minus the zero-grade students

X = df.drop(columns="G3")
y = df["G3"]
X_early = X.drop(columns=["G1", "G2"])    # for early-warning use
print("Features, all:", X.shape[1], "| early (no G1/G2):", X_early.shape[1])

Shape: (357, 42)
Features, all: 41 | early (no G1/G2): 39


In [5]:
# Cell 2: the train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Average G3, train:", round(y_train.mean(), 2), "| test:", round(y_test.mean(), 2))

# same rows, early-warning feature set
X_train_e = X_train.drop(columns=["G1", "G2"])
X_test_e = X_test.drop(columns=["G1", "G2"])

Train: (285, 41) | Test: (72, 41)
Average G3, train: 11.55 | test: 11.43


In [7]:
# Cell 3: a helper that reports train and test performance
def report(model, Xtr, Xte, name):
    model.fit(Xtr, y_train)
    rows = {}
    for label, Xs, ys in [("train", Xtr, y_train), ("test", Xte, y_test)]:
        pred = model.predict(Xs)
        rows[label] = {"RMSE": round(np.sqrt(mean_squared_error(ys, pred)), 2),
                       "R2": round(r2_score(ys, pred), 3)}
    print(f"{name:34s} train RMSE {rows['train']['RMSE']:5.2f} R2 {rows['train']['R2']:6.3f}"
          f" | test RMSE {rows['test']['RMSE']:5.2f} R2 {rows['test']['R2']:6.3f}")

In [8]:
# Cell 4: overfitting, seen with your own eyes (early-warning features)
report(LinearRegression(), X_train_e, X_test_e, "Linear regression")
report(DecisionTreeRegressor(random_state=42), X_train_e, X_test_e, "Deep decision tree (no limit)")
report(DecisionTreeRegressor(max_depth=3, random_state=42), X_train_e, X_test_e, "Shallow tree (max_depth=3)")

Linear regression                  train RMSE  2.62 R2  0.349 | test RMSE  2.73 R2  0.227
Deep decision tree (no limit)      train RMSE  0.00 R2  1.000 | test RMSE  3.53 R2 -0.292
Shallow tree (max_depth=3)         train RMSE  2.87 R2  0.222 | test RMSE  2.88 R2  0.138


In [9]:
# Cell 5: data leakage, the wrong way vs the right way
# WRONG: scaler sees every student, including the test ones
scaler_wrong = StandardScaler().fit(X[["absences"]])
# RIGHT: scaler learns only from training students
scaler_right = StandardScaler().fit(X_train[["absences"]])

print("What each scaler learned about absences")
print(f"  WRONG (all data)    mean {scaler_wrong.mean_[0]:.4f}  std {scaler_wrong.scale_[0]:.4f}")
print(f"  RIGHT (train only)  mean {scaler_right.mean_[0]:.4f}  std {scaler_right.scale_[0]:.4f}")

test_scaled = scaler_right.transform(X_test[["absences"]])
print("\nTest set after the RIGHT scaler: mean", round(test_scaled.mean(), 3),
      "(not exactly 0, and that is correct and honest)")

What each scaler learned about absences
  WRONG (all data)    mean 5.7395  std 5.7113
  RIGHT (train only)  mean 5.6351  std 5.7339

Test set after the RIGHT scaler: mean 0.09 (not exactly 0, and that is correct and honest)


In [10]:
# Cell 6: only scale numeric columns, not 0/1 columns
scale_cols = [c for c in X.columns if X[c].nunique() > 2]
binary_cols = [c for c in X.columns if X[c].nunique() <= 2]
print("Scaled columns:", len(scale_cols), "->", scale_cols)
print("Left as 0/1   :", len(binary_cols))

prep = ColumnTransformer(
    [("scale", StandardScaler(), scale_cols)],
    remainder="passthrough")

Scaled columns: 15 -> ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']
Left as 0/1   : 26


In [11]:
# Cell 7: a pipeline bundles scaling and the model, so leakage cannot happen
pipe = Pipeline([("prep", prep), ("model", LinearRegression())])

report(pipe, X_train, X_test, "Linear regression, all features")

Linear regression, all features    train RMSE  0.75 R2  0.946 | test RMSE  0.90 R2  0.916


In [12]:
# Cell 8: what scaling gives you, comparable coefficients
feature_names = scale_cols + binary_cols     # ColumnTransformer output order
coefs = pd.Series(pipe.named_steps["model"].coef_, index=feature_names)

print("Largest effects on G3 (scaled features, so directly comparable):")
print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(8).round(3))

Largest effects on G3 (scaled features, so directly comparable):
G2                2.803
guardian_other   -0.480
Mjob_teacher      0.428
G1                0.376
Pstatus          -0.296
Mjob_health       0.245
nursery          -0.233
Mjob_other       -0.221
dtype: float64


In [13]:
# Cell 9: cross-validation for a trustworthy score
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, Xtr in [("all features", X_train), ("early (no G1/G2)", X_train_e)]:
    sc_cols = [c for c in Xtr.columns if X[c].nunique() > 2]
    p = Pipeline([
        ("prep", ColumnTransformer([("s", StandardScaler(), sc_cols)], remainder="passthrough")),
        ("model", LinearRegression())])
    scores = -cross_val_score(p, Xtr, y_train, cv=cv, scoring="neg_root_mean_squared_error")
    print(f"{name:18s} RMSE per fold {scores.round(2)} | mean {scores.mean():.2f} +/- {scores.std():.2f}")

all features       RMSE per fold [0.96 0.92 0.84 0.79 1.01] | mean 0.91 +/- 0.08
early (no G1/G2)   RMSE per fold [3.12 3.09 3.16 2.72 3.17] | mean 3.05 +/- 0.17


In [14]:
# Cell 10: how much does the test score depend on luck? (different random splits)
results = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    p = Pipeline([("prep", prep), ("model", LinearRegression())]).fit(Xtr, ytr)
    results.append(np.sqrt(mean_squared_error(yte, p.predict(Xte))))

print("Test RMSE across 10 different splits:", np.round(results, 2))
print(f"Range: {min(results):.2f} to {max(results):.2f}")

Test RMSE across 10 different splits: [0.96 0.88 0.85 0.88 0.85 0.85 0.89 0.84 0.87 0.97]
Range: 0.84 to 0.97


In [15]:
# Cell 11: save the official split used for the rest of the project
train = X_train.copy(); train["G3"] = y_train
test = X_test.copy();  test["G3"] = y_test
train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)
print("Saved train.csv", train.shape, "and test.csv", test.shape)

Saved train.csv (285, 42) and test.csv (72, 42)
